In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import logging
import os
from pathlib import Path
import anndata2ri
from bokeh.models import TabPanel, Tabs,ColorBar
from bokeh.plotting import show, output_file
from scipy.stats import median_abs_deviation
import seaborn as sns
import scrublet as scr
import seaborn as sns
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
from utils import interactive_embedding
import anndata as ad
from scipy.sparse import csr_matrix, issparse
from bokeh.plotting import show, output_file, output_notebook

anndata2ri.activate()
%load_ext rpy2.ipython

# rcb.logger.setLevel(logging.ERROR)

sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
    figsize = (8,5)
)

pd.set_option("display.max_columns", None)
sc.settings.verbosity = 1
output_notebook()
%matplotlib inline


Loading BokehJS ...

In [2]:
sample_path = Path("../LBP_brain_blood_pairs/data/narsad_cellRanger_outs/blood/")

sample_list = []

sum_cells = 0
sum_genes = []
for i, sample in enumerate(sorted(os.listdir(sample_path))):
    if os.path.splitext(sample)[1] == '.h5ad':
        adata_sample = sc.read(sample_path / sample)
        adata_sample.obs['pt'] = '-'.join(sample.split('-')[:2])
        # adata_sample.obs['side'] = sample.split('-')[3].split("_")[0]
        sum_genes.extend(list(adata_sample.var.index))
        print(i,sample, adata_sample.shape)
        sample_list.append(adata_sample)
        sum_cells += adata_sample.shape[0]


2 PT-182-blood-R_QC.h5ad (4484, 15650)
4 PT-185-blood-L_QC.h5ad (8139, 15964)
6 PT-201-blood-L_QC.h5ad (3182, 17088)
8 PT-203-blood-L_QC.h5ad (2172, 17176)
10 PT-203-blood-R_QC.h5ad (4504, 19162)
12 PT-205-blood-L_QC.h5ad (2760, 18043)
14 PT-205-blood-R_QC.h5ad (3362, 17526)
16 PT-206-blood-L_QC.h5ad (1075, 15569)
18 PT-208-blood-L_QC.h5ad (2075, 15974)
20 PT-208-blood-R_QC.h5ad (2277, 15862)
22 PT-212-blood-L_QC.h5ad (993, 15709)
24 PT-212-blood-R_QC.h5ad (3293, 15765)
26 PT-214-blood-L_QC.h5ad (5243, 17186)
28 PT-214-blood-R_QC.h5ad (3527, 17627)


In [ ]:
sum_cells, len(pd.unique(sum_genes))

In [ ]:
adata_merged = ad.concat(sample_list, join="outer",  index_unique="_")
adata_merged.obs_names_make_unique()
print(adata_merged.X[0,:6])
adata_merged.X = adata_merged.layers['counts']
print(adata_merged.X[0,:6])

In [ ]:
adata_merged , adata_merged.shape,  adata_merged.X.shape,  adata_merged.layers['log1p_norm'].shape

In [ ]:
del sample_list

In [ ]:
scales_counts = sc.pp.normalize_total(adata_merged, layer='counts', exclude_highly_expressed=True, target_sum=None, inplace=False)
# log1p transform
adata_merged.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)


In [ ]:
labels = ['pt','side',"total_counts", 'outlier',"n_genes_by_counts", "pct_counts_mt", 'pct_counts_ribo', 'scDblFinder_score','scDblFinder_class','doublet_scores_scrublet', 'predicted_doublets_scrublet']
# sc.pp.normalize_total(adata_merged)
# sc.pp.log1p(adata_merged)
adata_log1p = adata_merged.copy()
adata_log1p.X = adata_log1p.layers["log1p_norm"]
sc.pp.pca(adata_log1p, svd_solver="arpack")
sc.pp.neighbors(adata_log1p)
sc.tl.umap(adata_log1p)
sc.pl.umap(adata_log1p, color=labels)

In [ ]:
adata_merged.obs['outlier'] = adata_merged.obs['outlier'].astype("category")

labels = ['pt',"log1p_total_counts",'log1p_n_genes_by_counts','pct_counts_in_top_20_genes', "outlier","pct_counts_mt", "pct_counts_ribo", "pct_counts_hb", "scDblFinder_score", "doublet_scores_scrublet", "scDblFinder_class", "predicted_doublets_scrublet"]

tabs = []

for label in labels:
    print(label)
    tabs.append(TabPanel(child=interactive_embedding(adata_log1p, label), title=label))

p = Tabs(tabs=tabs)
# output_file(f"../LBP_brain_blood_pairs/PLOTS/QC_plots/{PT}-blood-{side}_cellbender_QC.html")

# 
show(p)

In [ ]:
adata_filtered = adata_merged[(adata_merged.obs['predicted_doublets_scrublet'] == 'False') & (adata_merged.obs['pct_counts_mt']<20)].copy()

In [ ]:
adata_filtered.X = adata_filtered.layers['counts']
scales_counts = sc.pp.normalize_total(adata_filtered, layer='counts', exclude_highly_expressed=True, target_sum=None, inplace=False)
# log1p transform
adata_filtered.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)
adata_filtered.X = adata_filtered.layers["log1p_norm"]

In [ ]:
labels = ['pt',"side", "log1p_total_counts",'log1p_n_genes_by_counts','pct_counts_in_top_20_genes', "outlier","pct_counts_mt", "pct_counts_ribo", "pct_counts_hb", "scDblFinder_score"]

sc.pp.pca(adata_filtered, svd_solver="arpack")
sc.pp.neighbors(adata_filtered)
sc.tl.umap(adata_filtered)
sc.pl.umap(adata_filtered, color=labels)
tabs = []

for label in labels:
    tabs.append(TabPanel(child=interactive_embedding(adata_filtered, label), title=label))

p = Tabs(tabs=tabs)
# output_file(f"../LBP_brain_blood_pairs/PLOTS/QC_plots/{PT}-blood-{side}_cellbender_QC.html")

# 
show(p)

In [ ]:
%%R
library(scran)
library(BiocParallel)

In [ ]:
del adata_filtered

In [ ]:
adata_filtered = adata_merged[(adata_merged.obs['predicted_doublets_scrublet'] == 'False') & (adata_merged.obs['pct_counts_mt']<20)].copy()

In [ ]:
# Preliminary clustering for differentiated normalisation
adata_pp = adata_filtered.copy()
sc.pp.normalize_total(adata_pp)
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp, n_comps=15)
sc.pp.neighbors(adata_pp)
sc.tl.leiden(adata_pp, key_added="groups")

data_mat = adata_pp.X.T
# convert to CSC if possible. See https://github.com/MarioniLab/scran/issues/70
if issparse(data_mat):
    if data_mat.nnz > 2**31 - 1:
        data_mat = data_mat.tocoo()
    else:
        data_mat = data_mat.tocsc()
ro.globalenv["data_mat"] = data_mat
ro.globalenv["input_groups"] = adata_pp.obs["groups"]

del adata_pp


In [ ]:
%%R -o size_factors

size_factors = sizeFactors(
    computeSumFactors(
        SingleCellExperiment(
            list(counts=data_mat)), 
            clusters = input_groups,
            min.mean = 0.1,
            BPPARAM = MulticoreParam()
    )
)

In [ ]:
adata_filtered.obs["size_factors"] = size_factors
scran = adata_filtered.X / adata_filtered.obs["size_factors"].values[:, None]
adata_filtered.layers["scran_normalization"] = csr_matrix(sc.pp.log1p(scran))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
p1 = sns.histplot(adata_filtered.obs["total_counts"], bins=100, kde=False, ax=axes[0])
axes[0].set_title("Total counts")
p2 = sns.histplot(
    adata_filtered.layers["scran_normalization"].sum(1), bins=100, kde=False, ax=axes[1]
)
axes[1].set_title("log1p with Scran estimated size factors")
plt.show()

In [ ]:
print(adata_filtered.X[0,:6])
print(adata_log1p.X[0,:6])

In [ ]:
labels = ['pt',"side", "log1p_total_counts",'log1p_n_genes_by_counts','pct_counts_in_top_20_genes', "outlier","pct_counts_mt", "pct_counts_ribo", "pct_counts_hb", "scDblFinder_score"]
adata_filtered.X = adata_filtered.layers["scran_normalization"]
sc.pp.pca(adata_filtered, svd_solver="arpack")
sc.pp.neighbors(adata_filtered)
sc.tl.umap(adata_filtered)
sc.pl.umap(adata_filtered, color=labels)
tabs = []

for label in labels:
    tabs.append(TabPanel(child=interactive_embedding(adata_filtered, label), title=label))

p = Tabs(tabs=tabs)
# output_file(f"../LBP_brain_blood_pairs/PLOTS/QC_plots/{PT}-blood-{side}_cellbender_QC.html")

# 
show(p)

### Remove doublets

In [ ]:
#Remove doublets
# adata_b1 = adata_b1[adata_b1.obs['predicted_doublets_scrublet'] == False,:]
# adata_b1[adata_b1.obs['predicted_doublets_scrublet'] & adata_b1[adata_b1.obs['predicted_doublets_scrublet']
print(len(set(adata_b1.obs[adata_b1.obs["scDblFinder_class"]=='singlet'].index).intersection(adata_b1.obs[adata_b1.obs['predicted_doublets_scrublet']==False].index)))
print(len(set(adata_b1.obs[adata_b1.obs["scDblFinder_class"]=='singlet'].index)), len(set(adata_b1.obs[adata_b1.obs["predicted_doublets_scrublet"]==False].index)) )
adata_b1 = adata_b1[(adata_b1.obs["scDblFinder_class"]=='singlet') & (adata_b1.obs['predicted_doublets_scrublet']==False)]
adata_b1

### Filter mito genes

In [ ]:
mito_genes = adata_b1.var_names.str.startswith('MT-')
keep = np.invert(mito_genes)

adata_b1 = adata_b1[:,keep]
adata_b1

In [ ]:
sc.pp.normalize_total(adata_b1)
sc.pp.log1p(adata_b1)
sc.pp.highly_variable_genes(
    adata_b1, layer='counts', n_top_genes=5000, flavor="seurat_v3", batch_key='pt'
)
sc.pp.highly_variable_genes(
    adata_b1,  n_top_genes=5000
)
sc.pp.pca(adata_b1, svd_solver="arpack")


In [ ]:
# n_batches = adata_b1.var["highly_variable_nbatches"].value_counts()
# ax = n_batches.plot(kind="bar")
# n_batches

In [ ]:
sc.pp.neighbors(adata_b1)
sc.tl.umap(adata_b1)


In [ ]:
labels = ['pt','side',"total_counts", "n_genes_by_counts", "pct_counts_mt", 'pct_counts_ribo', 'scDblFinder_score','scDblFinder_class','doublet_scores_scrublet', 'predicted_doublets_scrublet']
sc.pl.umap(adata_b1, color=labels)

In [ ]:
adata_hvg = adata_b1[:, adata_b1.var["highly_variable"]].copy()
adata_hvg

In [ ]:
%matplotlib inline
adata_hvg.obs['scDblFinder_class'] = adata_hvg.obs['scDblFinder_class'].astype('str')
adata_hvg.obs['predicted_doublets_scrublet'] =adata_hvg.obs['predicted_doublets_scrublet'].astype('str')

labels = ['pt','side',"total_counts", "n_genes_by_counts", "pct_counts_mt", 'pct_counts_ribo', 'scDblFinder_score','scDblFinder_class','doublet_scores_scrublet', 'predicted_doublets_scrublet']
sc.pl.umap(adata_hvg, color=labels)
tabs = []



In [ ]:
## REMOVE MT GENES???

## Data integration

In [ ]:
import scvi
scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="pt")
model_scvi = scvi.model.SCVI(adata_hvg)


In [ ]:
# model_scvi.view_anndata_setup()


In [ ]:
max_epochs_scvi = np.min([round((20000 / adata_b1.n_obs) * 400), 400])
max_epochs_scvi

In [ ]:
model_scvi.train(use_gpu=False)


In [ ]:
model_scvi

In [ ]:
adata_scvi = adata_hvg.copy()
adata_scvi.obsm["X_scVI"] = model_scvi.get_latent_representation()
sc.pp.neighbors(adata_scvi, use_rep="X_scVI")
sc.tl.umap(adata_scvi)
adata_scvi

In [ ]:
# adata_scvi.write("../data/blood/blood_scvi_integration.h5ad")

In [ ]:
sc.pl.umap(adata_scvi, color=['pt', 'total_counts'], wspace=1)


In [ ]:
import scanpy.external as sce
adata_harmony = adata_hvg.copy()
sc.pp.pca(adata_harmony, svd_solver="arpack", use_highly_variable=False)

sce.pp.harmony_integrate(adata_harmony, "pt")
sc.pp.neighbors(adata_harmony, use_rep="X_pca_harmony")
sc.tl.umap(adata_harmony)
sc.pl.umap(adata_harmony, color=['pt', 'total_counts'], wspace=1)


In [ ]:
# adata_harmony.write("../data/blood/blood_harmony_integration.h5ad")

In [ ]:
%matplotlib inline
adata_hvg.obs['scDblFinder_class'] = adata_hvg.obs['scDblFinder_class'].astype('str')
adata_hvg.obs['predicted_doublets_scrublet'] =adata_hvg.obs['predicted_doublets_scrublet'].astype('str')

labels = ['pt','side',"total_counts", "n_genes_by_counts", "pct_counts_mt", 'pct_counts_ribo', 'scDblFinder_score','scDblFinder_class','doublet_scores_scrublet', 'predicted_doublets_scrublet']
sc.pl.umap(adata_hvg, color=labels)
tabs = []

from bokeh.models import TabPanel, Tabs,ColorBar
from bokeh.plotting import show, output_file
from utils import interactive_embedding

for label in labels:
    tabs.append(TabPanel(child=interactive_embedding(adata_hvg, label), title=label))

for label in ['pt','pct_counts_ribo', 'scDblFinder_score']:

    tabs.append(TabPanel(child=interactive_embedding(adata_scvi, label), title=f"SCVI integration - {label}"))
    tabs.append(TabPanel(child=interactive_embedding(adata_harmony, label), title=f"Harmony integration - {label}"))

# tabs.append(TabPanel(child=interactive_embedding(adata_seurat, "total_counts"), title="Seurat FS"))
# tabs.append(TabPanel(child=interactive_embedding(adata_seuratv3, "total_counts"), title="Seurat3 FS"))
# tabs.append(TabPanel(child=interactive_embedding(adata_pearson, "total_counts"), title="Pearson FS"))

p = Tabs(tabs=tabs)
output_file(f"../QC_plots/integrated-blood.html")

# 
show(p)

In [ ]:
%matplotlib inline
adata_hvg.obs['scDblFinder_class'] = adata_hvg.obs['scDblFinder_class'].astype('str')
adata_hvg.obs['predicted_doublets_scrublet'] =adata_hvg.obs['predicted_doublets_scrublet'].astype('str')

labels = ['pt','side',"total_counts", "n_genes_by_counts", "pct_counts_mt", 'pct_counts_ribo', 'scDblFinder_score','scDblFinder_class','doublet_scores_scrublet', 'predicted_doublets_scrublet']
sc.pl.umap(adata_hvg, color=labels)
tabs = []

from bokeh.models import TabPanel, Tabs,ColorBar
from bokeh.plotting import show, output_file
from utils import interactive_embedding

for label in labels:
    tabs.append(TabPanel(child=interactive_embedding(adata_hvg, label), title=label))

for label in ['pt','pct_counts_ribo', 'scDblFinder_score']:

    tabs.append(TabPanel(child=interactive_embedding(adata_scvi, "pt"), title="SCVI integration"))
    tabs.append(TabPanel(child=interactive_embedding(adata_harmony, "pt"), title="Harmony integration"))

# tabs.append(TabPanel(child=interactive_embedding(adata_seurat, "total_counts"), title="Seurat FS"))
# tabs.append(TabPanel(child=interactive_embedding(adata_seuratv3, "total_counts"), title="Seurat3 FS"))
# tabs.append(TabPanel(child=interactive_embedding(adata_pearson, "total_counts"), title="Pearson FS"))

p = Tabs(tabs=tabs)
output_file(f"../QC_plots/integrated-scvi-blood.html")

# 


## Compare methods

In [ ]:
# Bioilogical conservation metrics need cell label or other key
#metrics_scvi = scib.metrics.metrics_fast(
#     adata, adata_scvi, batch_key, label_key, embed="X_scVI"
# )

In [ ]:
import scib
# PCR_score: teh hogher the better, range 0-1
print("Unintegrated:", scib.metrics.pcr_comparison(adata_b1, adata_hvg, covariate="pt"), "\n")

print("Harmony",scib.metrics.pcr_comparison(adata_hvg, adata_harmony, covariate="pt", embed="X_pca_harmony"), "\n")
print("ScVI",scib.metrics.pcr_comparison(adata_hvg, adata_scvi, covariate="pt", embed="X_scVI"))

# Clustering

In [ ]:
import scvi
sc.tl.leiden(adata, key_added="leiden_res0_25", resolution=0.25)
sc.tl.leiden(adata, key_added="leiden_res0_5", resolution=0.5)
sc.tl.leiden(adata, key_added="leiden_res1", resolution=1.0)

In [ ]:
sc.tl.leiden(adata)
sc.pl.umap(
    adata_b1,
    color=["leiden_res0_25", "leiden_res0_5", "leiden_res1"],
    legend_loc="on data",
)